# Email Finder — End-to-End Test Notebook (EF-19)

Runs the full pipeline:
1. Define leads inline (or load from CSV / Google Sheet)
2. Discover emails via waterfall (`find_emails_batch`)
3. Export two split CSVs — enriched vs still needs enrichment

## Cell 1: Setup & Config

In [ ]:
import sys
import io
import textwrap
import logging
import pandas as pd
sys.path.append("../..")

from email_finder import (
    find_emails_batch,
    LeadInput,
    EmailFinderResult,
    load_leads_from_csv,
    load_leads_from_google_sheet,
    export_results,
)
from email_finder.config import Config

logging.basicConfig(level=logging.INFO)

config = Config()  # Loads from .env
print("Config loaded.")

## Cell 2: Define Test Leads Inline

Paste raw CSV text below. `Podcast Name` is used as `full_name` (the Perplexity node will look up the podcast host).  
To load from a file instead, use `load_leads_from_csv("path/to/file.csv")` in Cell 3.

In [ ]:
RAW_CSV = textwrap.dedent("""\
Podcast Name,Podcast Website,Podcast Email,Podcast Facebook,Podcast Twitter,Podcast Instagram,Podcast YouTube,Podcast LinkedIn
A Mommy And A Mic,http://www.amommyandamic.com/,podcast@myrockerbeez.com,,,,,
Culinary Treasure Podcast,https://www.culinarytreasurepodcast.com/,sshomler@me.com,https://www.facebook.com/CulinaryTreasurePodcast,,https://www.instagram.com/culinarytreasurepodcast,https://www.youtube.com/channel/UCA-zUuYpU_KQpVemaheeWEA,
Joy of Weightlessness,https://www.buzzsprout.com/2016334,,https://www.facebook.com/joalibng,https://twitter.com/JOALIBEING,https://www.instagram.com/joalibeing,,
TravelRight.Today,http://www.travelright.today/,,,,,,
Paper Trails,https://papertrails.podbean.com/,,,,,,
Waves of Impact,https://uwf.edu/commerce,,,,,,
The Smoking Barrel Podcast,http://thesmokingbarrelpodcast.com/,,https://www.facebook.com/thesmokingbarrelpodcast,,,,
The Southern Fork,http://www.thesouthernfork.com/episodes/,charlotteghost@gmail.com,,,"https://www.instagram.com/southernfork",,
Chef D's Bistro,https://podcasters.spotify.com/pod/show/darryl-ingram,darryl.ingram0162@gmail.com,,,,,
Grounded,https://www.groundedthepod.com/,"smoody09@gmail.com, tech@ringmaster.com",https://www.facebook.com/MichaelKLaRue,,https://www.instagram.com/tridavetri,,https://www.linkedin.com/in/amyhom17
""")

# The CSV has no Host Name column — we use Podcast Name as full_name.
# Perplexity will find the host from the podcast name + website.
COLUMN_MAPPING = {
    "Podcast Name":     "full_name",    # used as the search entity
    "Podcast Website":  "website",
    "Podcast Email":    "existing_email",
    "Podcast Facebook": "facebook_url",
    "Podcast Twitter":  "twitter_url",
    "Podcast Instagram":"instagram_url",
    "Podcast YouTube":  "youtube_url",
    "Podcast LinkedIn": "linkedin_url",
}

df = pd.read_csv(io.StringIO(RAW_CSV), dtype=str, keep_default_na=False)
df = df.replace("nan", "")
print(f"Parsed {len(df)} rows from inline CSV")
df

## Cell 3: Build LeadInput Objects

In [ ]:
from email_finder.io.loader import _row_to_lead

leads = []
for _, row in df.iterrows():
    lead = _row_to_lead(row, COLUMN_MAPPING)
    if lead:
        leads.append(lead)
    else:
        print(f"  Skipped row: {row.get('Podcast Name', '?')} (missing full_name)")

print(f"\nBuilt {len(leads)} LeadInput objects:")
for l in leads:
    print(f"  {l.full_name:<35}  email={l.existing_email or '—'}  website={l.website or '—'}")

## Cell 4: Run the Pipeline

In [ ]:
results = await find_emails_batch(leads, config)

## Cell 5: Preview Results

In [ ]:
status_icon = {
    "verified": "✓",
    "catch_all": "~",
    "unverified": "?",
    "not_found": "✗",
    "invalid": "✗",
}

print(f"{'Name':<35}  {'Email':<40}  {'Status':<12}  Conf")
print("-" * 100)
for lead, result in zip(leads, results):
    icon = status_icon.get(result.status, "?")
    email_str = result.email or "N/A"
    conf = f"{result.confidence:.0%}" if result.confidence else ""
    print(f"[{icon}] {lead.full_name:<33}  {email_str:<40}  {result.status:<12}  {conf}")

## Cell 6: Export to CSV

In [ ]:
files = export_results(
    leads,
    results,
    output_dir="./output",
    prefix="test_run",
)

print(f"\nEnriched:      {files['enriched']}")
print(f"Still needs:   {files['needs_enrichment']}")

## Cell 7: Debug — Inspect a Specific Lead's Discovery Log

In [ ]:
idx = 0  # Change index to inspect a different lead
lead = leads[idx]
result = results[idx]

print(f"Lead:       {lead.full_name}")
print(f"Email:      {result.email}")
print(f"Status:     {result.status}")
print(f"Confidence: {result.confidence:.0%}")
print(f"Source:     {result.source}")
print()
print("Discovery log:")
for entry in result.discovery_log:
    node = entry.get('node', '?')
    res = entry.get('result', {})
    found = res.get('found_email') or res.get('found_emails') or '—'
    err = res.get('error') or ''
    print(f"  [{node:<20}]  found={found}  {'ERROR: ' + err if err else ''}")

## Cell 8: Load from File (alternative to inline CSV)

Use this cell when reading from an actual CSV file instead of the inline definition above.

In [ ]:
# leads = load_leads_from_csv(
#     "path/to/needs_enrichment.csv",
#     column_mapping=COLUMN_MAPPING,  # reuse the same mapping from Cell 2
# )

# Or from Google Sheet:
# leads = load_leads_from_google_sheet(
#     "https://docs.google.com/spreadsheets/d/YOUR_SHEET_ID",
#     sheet_name="Needs_Enrichment",
#     column_mapping=COLUMN_MAPPING,
# )

print("Uncomment the loader above and re-run Cells 4–7")